# Resume Analyzer - NLP Resume & Job Description Matcher
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

A practical, transparent NLP project to match resumes against job descriptions. It compares document text similarity using **TF-IDF + Cosine Similarity** and verifies candidate competencies using a **custom skill-matching engine**.


## 1. Project Overview & Motivation

When hiring for technical roles, recruiters and candidates face a common challenge:
* Keyword matchers often fail on technical terms (for example, treating `C++` as `C`, or matching `Java` inside `JavaScript`).
* Using large language models (LLMs) for every resume can be slow, expensive, and prone to hallucinations. Plus, resumes contain sensitive personal data that shouldn't always be sent to 3rd-party APIs.

### What our solution does:
1. **Reads Resumes in Memory**: Extracts text from PDFs directly in memory without saving files to disk.
2. **Cleans Text Safely**: Normalizes text while explicitly preserving technical characters (`C++`, `C#`, `.NET`, `Node.js`, `REST APIs`).
3. **Extracts Real Skills**: Uses boundary-safe regex patterns to accurately catch 30+ technical skills without false positives.
4. **Calculates Vocabulary Similarity**: Uses TF-IDF with unigrams & bigrams (with sublinear frequency scaling to prevent keyword stuffing).
5. **Combines the Scores**: Calculates a final score (20% text similarity + 80% skill match) that is easy to understand and explain.


## 2. Setup & Dependencies

Let's install the required libraries:
* `pymupdf` (fitz) - Fast PDF text extraction in memory.
* `scikit-learn` - TF-IDF vectorization and cosine similarity.
* `pandas` & `numpy` - Data processing.
* `matplotlib` & `seaborn` - Visualization.


In [1]:
# Install dependencies
!pip install -q pymupdf scikit-learn pandas numpy matplotlib seaborn

import re
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Union, BinaryIO

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import fitz  # PyMuPDF

print("Libraries imported successfully!")
print(f"PyMuPDF version: {fitz.__version__}")


Libraries imported successfully!
PyMuPDF version: 1.25.3


## 3. PDF Text Extraction (In-Memory)

In production, saving uploaded resumes to disk creates security and privacy headaches. Here we read the PDF bytes straight from RAM, extract the text page by page, and close the file handle immediately.


In [2]:
def extract_text_from_pdf(pdf_source: Union[bytes, BinaryIO, io.BytesIO]) -> str:
    """Extract text from in-memory PDF bytes or a stream without writing to disk."""
    if pdf_source is None:
        raise ValueError("PDF source cannot be None.")

    # Read bytes from file object or direct bytes
    if hasattr(pdf_source, "read"):
        if hasattr(pdf_source, "seek"):
            pdf_source.seek(0)
        pdf_bytes = pdf_source.read()
    elif isinstance(pdf_source, (bytes, bytearray)):
        pdf_bytes = bytes(pdf_source)
    else:
        raise TypeError("Expected bytes or a binary stream.")

    if not pdf_bytes:
        raise ValueError("PDF data is empty (0 bytes).")

    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    try:
        if len(doc) == 0:
            raise ValueError("The PDF has 0 pages.")

        pages_text = []
        for page_idx in range(len(doc)):
            page_text = doc[page_idx].get_text("text").strip()
            if page_text:
                pages_text.append(page_text)

        full_text = "\n\n".join(pages_text).strip()
        if not full_text:
            raise ValueError("No extractable text found (document might be scanned/image-only).")
        return full_text
    finally:
        doc.close()

# Quick test: create a 1-page sample PDF in memory and extract text from it
test_doc = fitz.open()
test_page = test_doc.new_page()
sample_resume_content = (
    "John Doe - Software Engineer\n"
    "Skills: Python, Docker, PostgreSQL, AWS, Git\n"
    "Experience: Built scalable backend services using Python and REST APIs."
)
test_page.insert_text((50, 72), sample_resume_content, fontsize=11)
sample_bytes = test_doc.write()
test_doc.close()

extracted = extract_text_from_pdf(sample_bytes)
print("Extracted text from test PDF:\n")
print(extracted)


Extracted text from test PDF:

John Doe - Software Engineer
Skills: Python, Docker, PostgreSQL, AWS, Git
Experience: Built scalable backend services using Python and REST APIs.


## 4. Text Preprocessing (Preserving Tech Terms)

Standard text cleaners often strip out symbols with regex like `re.sub(r'[^a-zA-Z]', ' ', text)`.
The problem:
* `C++` turns into `C`
* `C#` turns into `C`
* `.NET` turns into `NET`
* `Node.js` turns into `Node js`

Our preprocessor strips out bullet points, quotes, and punctuation while **keeping** the symbols that technical terms depend on (`+`, `#`, `.`, `/`, `-`).


In [3]:
def preprocess_text(text: Optional[str]) -> str:
    """Clean formatting noise while preserving symbols for tech terms."""
    if not text or not isinstance(text, str):
        return ""

    # 1. Remove bullet points and special glyphs
    bullet_pattern = r"[\u2022\u2023\u25E6\u2043\u2219\u25CB\u25CF\u25AA\u25A0\u25C6\u2605\u2713\u2714\u25BA·*]"
    cleaned = re.sub(bullet_pattern, " ", text)

    # 2. Replace newlines and tabs with spaces
    cleaned = re.sub(r"[\r\n\t]+", " ", cleaned).lower()

    # 3. Remove brackets, quotes, and punctuation (keeps +, #, ., -, /)
    for ch in ['(', ')', '[', ']', '{', '}', '<', '>', '"', "'", '`', '~', ';', '!', '?', '|']:
        cleaned = cleaned.replace(ch, ' ')

    # 4. Clean commas and colons acting as item separators
    cleaned = re.sub(r"[,:](?=\s|$)", " ", cleaned)
    cleaned = re.sub(r"(?<=\s)[,:](?=\s|$)", " ", cleaned)

    # 5. Remove trailing sentence periods without breaking .NET or Node.js
    cleaned = re.sub(r"\.+(?=\s|$)", "", cleaned)

    # 6. Remove isolated stray symbols
    cleaned = re.sub(r"(?<=\s)[-–—\/%]+(?=\s|$)", " ", cleaned)
    cleaned = re.sub(r"^[-–—\/%]+(?=\s|$)", " ", cleaned)

    # 7. Collapse extra spaces
    return re.sub(r"\s+", " ", cleaned).strip()

# Let's test with real messy text containing tech tokens
messy_text = "• Technologies: C++, C#, .NET, Python, and Node.js. Built REST APIs! CI/CD on Linux."
clean_result = preprocess_text(messy_text)
print("Original:   ", messy_text)
print("Cleaned:    ", clean_result)


Original:    • Technologies: C++, C#, .NET, Python, and Node.js. Built REST APIs! CI/CD on Linux.
Cleaned:     technologies c++ c# .net python and node.js built rest apis ci/cd on linux


## 5. Technical Skill Extraction (Deterministic Regex)

Instead of relying on an LLM that might hallucinate skills or cost money, we use a curated dictionary of 30+ skills.

Notice the regex patterns use **lookarounds and word boundaries**:
* Searching for `Java` won't match `JavaScript`.
* Searching for `C` won't match `C++` or `C#`.
* Searching for `Express` won't trigger on everyday phrases like `"express interest"`.
* Standalone words like `"next"` or `"node"` only trigger when accompanied by `.js`.


In [4]:
# Curated skill patterns with boundary protection
SKILL_PATTERNS = {
    "Python": r"(?<![a-zA-Z0-9])python(?![a-zA-Z0-9])",
    "Java": r"(?<![a-zA-Z0-9._-])java(?![a-zA-Z0-9])",
    "JavaScript": r"(?<![a-zA-Z0-9._-])(?:javascript|js)(?![a-zA-Z0-9])",
    "TypeScript": r"(?<![a-zA-Z0-9._-])typescript(?![a-zA-Z0-9])",
    "C": r"(?<![a-zA-Z0-9+#\-_])c(?![a-zA-Z0-9+#])",
    "C++": r"(?<![a-zA-Z0-9])c\+\+(?![a-zA-Z0-9+])",
    "C#": r"(?<![a-zA-Z0-9])(?:c#|c\s*sharp)(?![a-zA-Z0-9#])",
    ".NET": r"(?<![a-zA-Z0-9])(?:\.net|dotnet|asp\.net)(?![a-zA-Z0-9])",
    "React": r"(?<![a-zA-Z0-9])react(?:\.js|js)?(?![a-zA-Z0-9])",
    "Next.js": r"(?<![a-zA-Z0-9])next(?:\.js|js|\s+js)(?![a-zA-Z0-9])",
    "Node.js": r"(?<![a-zA-Z0-9])node(?:\.js|js|\s+js)(?![a-zA-Z0-9])",
    "Express": r"(?<![a-zA-Z0-9])(?:express(?:\.js|js)|express\s+framework|express\s+backend|\bexpress\b(?!\s+(?:interest|gratitude|enthusiasm|opinion|concerns|desire|myself|ourselves|ideas|clearly|an?\b|the\b|my\b|our\b|your\b)))(?![a-zA-Z0-9])",
    "SQL": r"(?<![a-zA-Z0-9])sql(?![a-zA-Z0-9])",
    "MySQL": r"(?<![a-zA-Z0-9])mysql(?![a-zA-Z0-9])",
    "PostgreSQL": r"(?<![a-zA-Z0-9])(?:postgresql|postgres)(?![a-zA-Z0-9])",
    "MongoDB": r"(?<![a-zA-Z0-9])(?:mongodb|mongo)(?![a-zA-Z0-9])",
    "Machine Learning": r"(?<![a-zA-Z0-9])machine\s+learning(?![a-zA-Z0-9])",
    "Deep Learning": r"(?<![a-zA-Z0-9])deep\s+learning(?![a-zA-Z0-9])",
    "NLP": r"(?<![a-zA-Z0-9])(?:nlp|natural\s+language\s+processing)(?![a-zA-Z0-9])",
    "Data Structures": r"(?<![a-zA-Z0-9])data\s+structures?(?![a-zA-Z0-9])",
    "Algorithms": r"(?<![a-zA-Z0-9])algorithms?(?![a-zA-Z0-9])",
    "Git": r"(?<![a-zA-Z0-9])git(?![a-zA-Z0-9])",
    "Docker": r"(?<![a-zA-Z0-9])docker(?![a-zA-Z0-9])",
    "REST API": r"(?<![a-zA-Z0-9])(?:rest\s*apis?|restful\s*apis?|rest\s+endpoints?)(?![a-zA-Z0-9])",
    "Pandas": r"(?<![a-zA-Z0-9])pandas(?![a-zA-Z0-9])",
    "NumPy": r"(?<![a-zA-Z0-9])numpy(?![a-zA-Z0-9])",
    "TensorFlow": r"(?<![a-zA-Z0-9])tensorflow(?![a-zA-Z0-9])",
    "PyTorch": r"(?<![a-zA-Z0-9])pytorch(?![a-zA-Z0-9])",
    "AWS": r"(?<![a-zA-Z0-9])(?:aws|amazon\s+web\s+services)(?![a-zA-Z0-9])",
    "Linux": r"(?<![a-zA-Z0-9])linux(?![a-zA-Z0-9])",
}

# Precompile regex patterns once for fast matching
COMPILED_PATTERNS = {
    skill: re.compile(pat, re.IGNORECASE) for skill, pat in SKILL_PATTERNS.items()
}

def extract_skills(text: Optional[str]) -> list[str]:
    """Extract recognized technical skills from text."""
    if not text or not isinstance(text, str):
        return []
    found = [skill for skill, rx in COMPILED_PATTERNS.items() if rx.search(text)]
    return sorted(found)

def compare_skills(resume_text: str, jd_text: str) -> dict:
    """Compare resume skills against JD requirements."""
    resume_skills = extract_skills(resume_text)
    required_skills = extract_skills(jd_text)

    r_set = set(resume_skills)
    j_set = set(required_skills)

    matched = sorted(list(r_set.intersection(j_set)))
    missing = sorted(list(j_set.difference(r_set)))

    return {
        "resume_skills": resume_skills,
        "required_skills": required_skills,
        "matched_skills": matched,
        "missing_skills": missing,
    }

# Boundary check test: ensure 'Java' is not detected when 'JavaScript' is written
tricky_sample = "I want to express interest. I develop in JavaScript, build APIs in C++, and use PostgreSQL."
print("Sample text:", tricky_sample)
print("Detected skills:", extract_skills(tricky_sample))


Sample text: I want to express interest. I develop in JavaScript, build APIs in C++, and use PostgreSQL.
Detected skills: ['C++', 'JavaScript', 'PostgreSQL', 'REST API']


## 6. Text Similarity with TF-IDF & Cosine Similarity

Next, we evaluate overall vocabulary and phrasing overlap.

Why these settings:
* **`ngram_range=(1, 2)`**: Catches both individual words (`"docker"`) and 2-word phrases (`"machine learning"`, `"rest api"`).
* **`sublinear_tf=True`**: Uses $1 + \log(\text{tf})$ instead of raw frequency, so repeated keyword stuffing does not inflate the score.
* **`token_pattern=r"(?u)\S+"`**: Treats any non-space sequence as a token so words like `c++` or `.net` stay intact.


In [5]:
def compute_text_similarity(resume_text: Optional[str], jd_text: Optional[str]) -> float:
    """Compute cosine similarity score between resume and JD (0.0 to 100.0)."""
    clean_resume = preprocess_text(resume_text)
    clean_jd = preprocess_text(jd_text)

    if not clean_resume or not clean_jd:
        return 0.0

    try:
        vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            sublinear_tf=True,
            token_pattern=r"(?u)\S+",
        )
        tfidf_matrix = vectorizer.fit_transform([clean_resume, clean_jd])
        sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0] * 100.0

        if np.isnan(sim):
            return 0.0
        return round(float(np.clip(sim, 0.0, 100.0)), 2)
    except ValueError:
        return 0.0

# Quick test with two related sentences
test_r = "Backend engineer experienced with Python, Docker, and REST APIs on AWS."
test_j = "Hiring a backend developer skilled in Python, Docker, and cloud AWS deployments."
sim_score = compute_text_similarity(test_r, test_j)
print(f"Similarity Score: {sim_score:.2f}%")


Similarity Score: 39.54%


## 7. Scoring Logic & Match Evaluation

We calculate two main scores and combine them:

1. **Skill Match Score**:
   $$\text{Skill Match (\%)} = \frac{\text{Matched Skills}}{\text{Required Skills in JD}} \times 100$$
   *Note: If the Job Description mentions zero recognized skills, we return `None` (Unavailable) so the candidate isn't penalized with 0% due to an unspecific JD.*

2. **Combined Score**:
   $$\text{Combined Score} = (0.20 \times \text{Text Similarity}) + (0.80 \times \text{Skill Match})$$
   *We give 80% weight to concrete skill coverage and 20% to vocabulary overlap.*


In [6]:
def evaluate_match(
    resume_text: str,
    jd_text: str,
    text_weight: float = 0.2,
    skill_weight: float = 0.8
) -> dict:
    """Run full match analysis between resume and job description."""
    text_sim = compute_text_similarity(resume_text, jd_text)
    skill_info = compare_skills(resume_text, jd_text)

    matched = skill_info["matched_skills"]
    required = skill_info["required_skills"]

    # Calculate skill match percentage
    if required:
        skill_match = round(float(np.clip((len(matched) / len(required)) * 100.0, 0.0, 100.0)), 2)
    else:
        skill_match = None  # JD did not specify technical skills from our dictionary

    # Calculate combined weighted score
    if skill_match is not None:
        total_w = text_weight + skill_weight
        w_text = text_weight / total_w
        w_skill = skill_weight / total_w
        combined = round(float(np.clip((w_text * text_sim) + (w_skill * skill_match), 0.0, 100.0)), 2)
    else:
        combined = None

    return {
        "text_similarity": text_sim,
        "skill_match": skill_match,
        "combined_score": combined,
        "resume_skills": skill_info["resume_skills"],
        "required_skills": skill_info["required_skills"],
        "matched_skills": matched,
        "missing_skills": skill_info["missing_skills"],
    }

print("Scoring engine ready.")


Scoring engine ready.


## 8. Testing on Real Candidate Scenarios

Let's test our pipeline on four diverse cases:
1. **Case 1 (Strong Overlap)**: Senior Backend Engineer matching a Cloud Backend role.
2. **Case 2 (Partial Overlap)**: Full-Stack Dev with some missing stack requirements (e.g. Docker, PostgreSQL).
3. **Case 3 (Unrelated)**: Executive Chef applying for an Embedded Robotics position.
4. **Case 4 (Behavioral JD)**: Candidate has data skills, but JD contains only soft/behavioral text.


In [7]:
test_scenarios = [
    {
        "name": "1. Strong Match (Backend)",
        "resume": (
            "Senior Backend Engineer with 6 years experience in Python, Docker, PostgreSQL, "
            "Git, Linux, and REST APIs. Experience building microservices on AWS and low latency C++."
        ),
        "jd": (
            "Seeking a Senior Backend Engineer proficient in Python, Docker, PostgreSQL, "
            "REST APIs, Git, and Linux. Familiarity with AWS and C++ is a plus."
        ),
    },
    {
        "name": "2. Partial Match (Full-Stack)",
        "resume": (
            "Full-Stack Web Developer skilled in JavaScript, React, Node.js, and MongoDB. "
            "Built client applications and integrated third-party REST APIs using Git."
        ),
        "jd": (
            "Looking for a Full-Stack Developer with experience in TypeScript, React, Next.js, "
            "Docker, PostgreSQL, Git, and AWS cloud deployments."
        ),
    },
    {
        "name": "3. Unrelated (Chef vs Robotics)",
        "resume": (
            "Executive Chef with 10 years experience in kitchen management, fine dining menu design, "
            "food safety sanitation, staff supervision, and vendor negotiations."
        ),
        "jd": (
            "Embedded Robotics Engineer needed for autonomous drone firmware. "
            "Requirements: C, C++, Linux device drivers, RTOS, and Git version control."
        ),
    },
    {
        "name": "4. Behavioral JD (No Tech Skills)",
        "resume": (
            "Data Scientist skilled in Python, Machine Learning, Deep Learning, Pandas, and NumPy."
        ),
        "jd": (
            "Looking for an energetic, collaborative team player with excellent communication skills, "
            "active listening, and strong presentation abilities."
        ),
    },
]

results = []
for case in test_scenarios:
    res = evaluate_match(case["resume"], case["jd"])
    results.append({
        "Scenario": case["name"],
        "Text Sim (%)": res["text_similarity"],
        "Skill Match (%)": f"{res['skill_match']:.1f}%" if res["skill_match"] is not None else "N/A",
        "Combined (%)": f"{res['combined_score']:.1f}%" if res["combined_score"] is not None else "N/A",
        "Matched Skills": ", ".join(res["matched_skills"]) if res["matched_skills"] else "None",
        "Missing Skills": ", ".join(res["missing_skills"]) if res["missing_skills"] else "None",
    })

df_results = pd.DataFrame(results)
df_results


                               Scenario  Text Sim (%) Skill Match (%) Combined (%)                                     Matched Skills                                     Missing Skills
0              1. Strong Match (Backend)         39.38          100.0%        87.9%  AWS, C++, Docker, Git, Linux, PostgreSQL, Pyth...                                               None
1          2. Partial Match (Full-Stack)         14.88           28.6%        25.8%                                         Git, React               AWS, Docker, Next.js, PostgreSQL, TypeScript
2         3. Unrelated (Chef vs Robotics)          0.00            0.0%         0.0%                                               None                                  C, C++, Git, Linux
3        4. Behavioral JD (No Tech Skills)          0.00             N/A          N/A                                               None                                               None


## 9. Visualizing Results & Gap Analysis

Let's generate two plots:
1. **Metric Comparison Across Scenarios** (shows how scores clearly separate strong, partial, and unrelated matches).
2. **Skill Gap Breakdown** for the partial match scenario (highlighting matched vs missing requirements).


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Comparison Bar Chart
labels = ["Strong Match", "Partial Match", "Unrelated", "Non-Tech JD"]
text_scores = [df_results.iloc[i]["Text Sim (%)"] for i in range(4)]
skill_scores = [
    evaluate_match(c["resume"], c["jd"])["skill_match"] or 0.0 for c in test_scenarios
]
combined_scores = [
    evaluate_match(c["resume"], c["jd"])["combined_score"] or 0.0 for c in test_scenarios
]

x = np.arange(len(labels))
width = 0.25

rects1 = axes[0].bar(x - width, text_scores, width, label="Text Similarity", color="#2563eb")
rects2 = axes[0].bar(x, skill_scores, width, label="Skill Match", color="#059669")
rects3 = axes[0].bar(x + width, combined_scores, width, label="Combined Score", color="#7c3aed")

axes[0].set_title("Scores Across Benchmark Scenarios", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Score (%)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, fontsize=10)
axes[0].set_ylim(0, 110)
axes[0].legend(loc="upper right")

for group in [rects1, rects2, rects3]:
    for rect in group:
        h = rect.get_height()
        if h > 0:
            axes[0].annotate(f"{h:.1f}%",
                xy=(rect.get_x() + rect.get_width() / 2, h),
                xytext=(0, 2), textcoords="offset points",
                ha="center", va="bottom", fontsize=8)

# Plot 2: Donut Chart for Partial Match Case
partial_case = evaluate_match(test_scenarios[1]["resume"], test_scenarios[1]["jd"])
matched_n = len(partial_case["matched_skills"])
missing_n = len(partial_case["missing_skills"])

axes[1].pie(
    [matched_n, missing_n],
    labels=[f"Matched ({matched_n})", f"Missing ({missing_n})"],
    autopct="%1.1f%%",
    colors=["#10b981", "#ef4444"],
    startangle=90,
    wedgeprops=dict(width=0.45, edgecolor="white")
)
axes[1].set_title("Skill Coverage for Scenario 2 (Partial Match)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()


## 10. Interactive Test (Try your own Resume & JD)

Paste any custom resume text and job description below to analyze alignment instantly.


In [9]:
# Modify these strings to test any custom resume and JD:
my_resume = """
Alex Sharma - Machine Learning Engineer
Skills: Python, PyTorch, TensorFlow, Pandas, NumPy, Docker, Git, Linux
Experience: Built deep learning recommendation systems and deployed REST APIs using Docker on AWS.
"""

my_jd = """
We are hiring a Machine Learning Engineer.
Requirements:
- Strong programming in Python.
- Hands-on experience with PyTorch or TensorFlow for Deep Learning.
- Experience containerizing models using Docker.
- Working knowledge of Git, Linux, and REST APIs.
- AWS experience is a plus.
"""

analysis = evaluate_match(my_resume, my_jd)

print("=" * 55)
print("              MATCH ANALYSIS REPORT")
print("=" * 55)
print(f"• Text Similarity:     {analysis['text_similarity']:.2f}%")
print(f"• Skill Match:         {analysis['skill_match']:.2f}%")
print(f"• Final Combined Score:{analysis['combined_score']:.2f}%")
print("-" * 55)
print(f"Matched Skills ({len(analysis['matched_skills'])}):")
print(" ", ", ".join(analysis['matched_skills']))
print(f"\nMissing Skills ({len(analysis['missing_skills'])}):")
print(" ", ", ".join(analysis['missing_skills']) if analysis['missing_skills'] else "None (100% requirements matched!)")
print(f"\nOther Skills on Resume:")
other = [s for s in analysis['resume_skills'] if s not in analysis['matched_skills']]
print(" ", ", ".join(other) if other else "None")
print("=" * 55)


              MATCH ANALYSIS REPORT
• Text Similarity:     45.24%
• Skill Match:         88.89%
• Final Combined Score:80.16%
-------------------------------------------------------
Matched Skills (8):
  AWS, Docker, Git, Linux, Machine Learning, PyTorch, Python, REST API

Missing Skills (1):
  Deep Learning

Other Skills on Resume:
  NumPy, Pandas, TensorFlow


## 11. Full Web App (Streamlit Dashboard)

In addition to this Colab notebook, our project includes a Streamlit web interface in `app.py`.
It provides:
* Drag-and-drop PDF resume upload.
* Job description input area with live analysis.
* SVG radial score gauge and visual skill pills (green for matched, red for missing).
* Direct resume quote extraction showing where each skill was identified.

To run the web app locally:
```bash
streamlit run app.py
```


## 12. Limitations & Practical Considerations

Being honest about a project's limitations is important:
1. **Curated Skill Dictionary**: Right now, the skill matcher uses a 30-skill dictionary. If a job requires something outside the dictionary (e.g. `Kubernetes`, `Rust`, or `GCP`), it will still help the TF-IDF text similarity score, but it won't show up in the skill pills until added to the regex dictionary.
2. **Scanned/Image PDFs**: `PyMuPDF` extracts digital text layers. If a resume is a scanned image (photo of paper), it requires an OCR tool like `Tesseract` before processing.
3. **Synonyms & Paraphrasing**: TF-IDF checks exact word stems and bigrams. For deeper semantic understanding (e.g. recognizing that *"managed database replicas"* relates to *"high availability"*), adding a dense embedding model like Sentence-Transformers (`all-MiniLM-L6-v2`) would be the natural next step.


## 13. Conclusion

### Summary:
* We built an end-to-end, explainable Resume Analyzer that runs fast and completely in memory.
* By separating **vocabulary overlap (TF-IDF)** from **hard competency requirements (Regex Skill Engine)**, we get a much clearer picture than a single naive keyword counter.
* The system protects user privacy by avoiding external API calls and never saving resume files to disk.
* All scoring logic is transparent and configurable, avoiding black-box rejection traps.

Thank you for checking out the project!
